# KDD Process: Credit Card Fraud DetectionDataset: [Credit Card Fraud Detection - Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)We apply the **KDD** lifecycle focused on anomaly detection for fraudulent transactions.

## KDD Phases- Selection- Preprocessing- Transformation- Data Mining- Interpretation & Evaluation

## Google Colab Setup

**Running in Colab?** This cell auto-configures your environment.

**Running locally?** Cell will be skipped automatically.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Handle Colab vs local paths
if IN_COLAB:
    try:
        transactions = pd.read_csv(filename)
        print(f"✅ Loaded dataset from Colab: {filename}")
    except NameError:
        print("⚠️  Please run one of the dataset loading options above (A, B, or C)")
        raise FileNotFoundError("Dataset not loaded. See instructions above.")
else:
    dataset_path = Path('../data/raw/creditcard.csv')
    if not dataset_path.exists():
        raise FileNotFoundError(f'Place creditcard.csv under {dataset_path.parent} after downloading from Kaggle.')
    transactions = pd.read_csv(dataset_path)
    print(f"✅ Loaded dataset from local path: {dataset_path}")

print(f"\n📊 Dataset shape: {transactions.shape[0]:,} rows × {transactions.shape[1]} columns")
transactions.head()

### Dataset Loading for Colab

**Choose ONE method if in Colab:**

**Option A: Direct Upload**
```python
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
```

**Option B: Kaggle API**
```python
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -q creditcardfraud.zip
filename = 'creditcard.csv'
```

**Option C: Google Drive**
```python
from google.colab import drive
drive.mount('/content/drive')
filename = '/content/drive/My Drive/DS-Methodologies/data/creditcard.csv'
```

**Local users:** Skip this - default path will be used.

## Selection**Objective**: Ingest transaction records with minimal leakage and design sampling strategies for the severe class imbalance.**Source**: Kaggle credit card fraud dataset (284,807 rows, 492 fraud cases).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

dataset_path = Path('../data/raw/creditcard.csv')
if not dataset_path.exists():
    raise FileNotFoundError('Place creditcard.csv under data/raw after downloading from Kaggle.')

transactions = pd.read_csv(dataset_path)
transactions.head()


### Data Overview & Class Imbalance Analysis
Understanding the extreme imbalance is critical for KDD fraud detection.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print(f'Dataset Shape: {transactions.shape}')
print(f'\nBasic Statistics:')
print(transactions.describe())

# Class distribution
fraud_count = transactions['Class'].value_counts()
print(f'\n=== Class Distribution ===')
print(f'Legitimate Transactions: {fraud_count[0]:,} ({fraud_count[0]/len(transactions)*100:.2f}%)')
print(f'Fraudulent Transactions: {fraud_count[1]:,} ({fraud_count[1]/len(transactions)*100:.4f}%)')
print(f'Imbalance Ratio: {fraud_count[0]/fraud_count[1]:.1f}:1')

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
ax[0].bar(['Legitimate', 'Fraud'], fraud_count.values, color=['#2ecc71', '#e74c3c'], alpha=0.7, edgecolor='black')
ax[0].set_ylabel('Count')
ax[0].set_title('Transaction Class Distribution', fontsize=12, fontweight='bold')
ax[0].set_yscale('log')  # Log scale to visualize both classes
ax[0].grid(axis='y', alpha=0.3)

# Add count labels
for i, count in enumerate(fraud_count.values):
    ax[0].text(i, count, f'{count:,}', ha='center', va='bottom', fontweight='bold')

# Amount distribution by class
transactions_fraud = transactions[transactions['Class'] == 1]
transactions_legit = transactions[transactions['Class'] == 0]

ax[1].hist(transactions_legit['Amount'], bins=50, alpha=0.6, label='Legitimate', color='#2ecc71', range=(0, 500))
ax[1].hist(transactions_fraud['Amount'], bins=50, alpha=0.6, label='Fraud', color='#e74c3c', range=(0, 500))
ax[1].set_xlabel('Transaction Amount ($)')
ax[1].set_ylabel('Frequency')
ax[1].set_title('Transaction Amount Distribution by Class', fontsize=12, fontweight='bold')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nFraud Transaction Statistics:')
print(f'Mean Amount: ${transactions_fraud["Amount"].mean():.2f}')
print(f'Median Amount: ${transactions_fraud["Amount"].median():.2f}')
print(f'Max Amount: ${transactions_fraud["Amount"].max():.2f}')

---

## 🤖 AI LEARNING CHECKPOINT #2: Preprocessing

**Pause here!** You've prepared your data splits.

### What You Just Learned
✅ Creating stratified train/validation/test splits  
✅ Preserving fraud ratio across all splits  
✅ Handling missing values (verification)  
✅ Maintaining temporal ordering in fraud data  

### Ask AI to Critique

```
You are a data preprocessing expert specializing in KDD methodology for fraud detection.
Review my Preprocessing phase.

CONTEXT:
- Missing values: [paste missing_summary output]
- Train/Valid/Test split: [paste shapes]
- Stratification: Yes, preserving 579:1 ratio

MY APPROACH:
[Paste split code from cell-8]

CRITIQUE QUESTIONS:
1. Is my train/valid/test split ratio (60/20/20) optimal for this imbalanced dataset?
2. Should I preserve temporal ordering more explicitly?
3. Are there preprocessing steps I should do before splitting?
4. What validation strategy best suits extreme imbalance?
5. Should I create a separate fraud-focused validation set?
6. How do I prevent data leakage in this scenario?

Provide 5-10 actionable improvements.
```

### Document AI Feedback

*[Paste AI response here]*

### Apply Improvements

**Possible enhancements:**
- Time-based splitting (instead of random)
- Cross-validation strategy for imbalanced data
- Fraud-specific validation metrics
- Data leakage checks

---

**Ready?** Continue to **Transformation** below.

---

---

## 🤖 AI LEARNING CHECKPOINT #1: Selection

**Pause here!** You've selected and loaded your dataset.

### What You Just Learned
✅ Understanding extreme class imbalance (579:1 ratio)  
✅ KDD Selection phase: choosing relevant data  
✅ Identifying the business context (fraud detection)  
✅ Dataset characteristics (284K transactions, 492 frauds)  

### Ask AI to Critique

```
You are a KDD (Knowledge Discovery in Databases) methodology expert 
specializing in fraud detection. Review my Selection phase.

CONTEXT:
- Methodology: KDD Process
- Project: Credit Card Fraud Detection
- Dataset: 284,807 transactions, 492 frauds (0.17% fraud rate)
- Imbalance ratio: 579:1

MY ANALYSIS:
[Paste class distribution and imbalance analysis output]

CRITIQUE QUESTIONS:
1. Is my understanding of the class imbalance problem complete?
2. Should I consider different sampling strategies in the Selection phase?
3. Are there data quality checks I should perform at this stage?
4. How should the extreme imbalance influence my KDD approach?
5. What business context am I missing about fraud detection?
6. Should I select a subset of the data for initial exploration?

Provide 5-10 actionable improvements for this Selection phase.
```

### Document AI Feedback

*[Paste AI insights here]*

### Apply Improvements

**Potential actions:**
- Stratified sampling for faster experimentation
- Business cost analysis setup
- Additional data quality checks
- Timeline validation (temporal ordering)

---

**Ready?** Continue to **Preprocessing** below.

---

## Preprocessing- Ensure time ordering is respected.- Handle missing values (dataset should be clean but verify).- Create a stratified train/validation/test split that preserves fraud ratio.

In [ ]:
missing_summary = transactions.isna().sum()
missing_summary[missing_summary > 0]


---

## 🤖 AI LEARNING CHECKPOINT #3: Transformation

**Pause here!** You've transformed features and analyzed patterns.

### What You Just Learned
✅ Feature scaling (StandardScaler for Amount and Time)  
✅ Temporal pattern analysis (fraud by hour)  
✅ Creating derived features for anomaly detection  
✅ Understanding fraud distribution over time  

### Ask AI to Critique

```
You are a feature engineering expert specializing in KDD and anomaly detection.
Review my Transformation phase for credit card fraud.

CONTEXT:
- Scaled features: Amount, Time
- Temporal analysis: Fraud rate by hour
- PCA features: V1-V28 (already transformed by dataset provider)

MY WORK:
[Paste scaling code and temporal analysis insights]

CRITIQUE QUESTIONS:
1. Is StandardScaler appropriate for this fraud detection use case?
2. What additional features could I engineer from Time and Amount?
3. Should I create rolling aggregates or velocity features?
4. How can I better leverage the temporal patterns I found?
5. Are there interaction features worth creating?
6. Should I transform the PCA features further?
7. What about feature selection - are all features necessary?

Provide 5-10 actionable improvements for transformation.
```

### Document AI Feedback

*[Paste AI recommendations here]*

### Apply Improvements

**Feature engineering ideas:**
- Rolling window aggregations (transaction velocity)
- Hour-of-day bins based on fraud patterns
- Amount percentile features
- Time since last transaction
- Feature interactions

---

**Ready?** Continue to **Data Mining** below.

---

In [ ]:
from sklearn.model_selection import train_test_split

train_val, test = train_test_split(
    transactions, test_size=0.2, stratify=transactions['Class'], random_state=42)
train, valid = train_test_split(
    train_val, test_size=0.25, stratify=train_val['Class'], random_state=42)

train.shape, valid.shape, test.shape


## Transformation**Actions**- Scale `Amount` and `Time`.- Engineer rolling aggregates for transaction velocity.- Address class imbalance via SMOTE or `class_weight` adjustments.

### Temporal Pattern Analysis
Analyze fraud patterns over time to identify high-risk periods.

In [ ]:
# Temporal analysis
fraud_by_hour = train.groupby(['hour', 'Class']).size().unstack(fill_value=0)
fraud_rate_by_hour = (fraud_by_hour[1] / fraud_by_hour.sum(axis=1) * 100)

fig, ax = plt.subplots(2, 2, figsize=(15, 10))

# Fraud transactions over time
ax[0, 0].scatter(train[train['Class']==0]['Time']/3600, train[train['Class']==0]['Amount'], 
                 alpha=0.01, s=1, c='#2ecc71', label='Legitimate')
ax[0, 0].scatter(train[train['Class']==1]['Time']/3600, train[train['Class']==1]['Amount'], 
                 alpha=0.8, s=20, c='#e74c3c', label='Fraud', edgecolors='black', linewidth=0.5)
ax[0, 0].set_xlabel('Time (hours)')
ax[0, 0].set_ylabel('Transaction Amount ($)')
ax[0, 0].set_title('Transaction Patterns Over Time', fontsize=12, fontweight='bold')
ax[0, 0].legend()
ax[0, 0].set_ylim(0, 1000)

# Fraud rate by hour
ax[0, 1].bar(fraud_rate_by_hour.index, fraud_rate_by_hour.values, color='#e74c3c', alpha=0.7, edgecolor='black')
ax[0, 1].set_xlabel('Hour of Day')
ax[0, 1].set_ylabel('Fraud Rate (%)')
ax[0, 1].set_title('Fraud Rate by Hour', fontsize=12, fontweight='bold')
ax[0, 1].grid(axis='y', alpha=0.3)

# Transaction volume by hour
tx_by_hour = train.groupby('hour').size()
ax[1, 0].plot(tx_by_hour.index, tx_by_hour.values, marker='o', linewidth=2, markersize=6, color='#3498db')
ax[1, 0].set_xlabel('Hour of Day')
ax[1, 0].set_ylabel('Transaction Count')
ax[1, 0].set_title('Transaction Volume by Hour', fontsize=12, fontweight='bold')
ax[1, 0].grid(alpha=0.3)

# Amount statistics by class
amount_stats = train.groupby('Class')['Amount'].describe()[['mean', '50%', '75%', 'max']]
amount_stats.plot(kind='bar', ax=ax[1, 1], color=['#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'], alpha=0.7)
ax[1, 1].set_xlabel('Class (0=Legit, 1=Fraud)')
ax[1, 1].set_ylabel('Amount ($)')
ax[1, 1].set_title('Amount Statistics by Class', fontsize=12, fontweight='bold')
ax[1, 1].legend(['Mean', 'Median', '75th %ile', 'Max'], loc='upper right')
ax[1, 1].tick_params(axis='x', rotation=0)
ax[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('\\n=== Key Temporal Insights ===')
print(f'Peak fraud hour: {fraud_rate_by_hour.idxmax()} (fraud rate: {fraud_rate_by_hour.max():.4f}%)')
print(f'Safest hour: {fraud_rate_by_hour.idxmin()} (fraud rate: {fraud_rate_by_hour.min():.4f}%)')

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
for frame in (train, valid, test):
    frame['scaled_amount'] = scaler.fit_transform(frame[['Amount']]) if frame is train else scaler.transform(frame[['Amount']])
    frame['scaled_time'] = scaler.fit_transform(frame[['Time']]) if frame is train else scaler.transform(frame[['Time']])


### Advanced Anomaly Detection Models
Compare specialized algorithms for rare-event detection including ensemble methods and neural approaches.

In [ ]:
# Cost-Sensitive Threshold Optimization
# Business costs (hypothetical but realistic)
cost_false_positive = 5  # Cost of investigating legitimate transaction
cost_false_negative = 200  # Average fraud loss per missed fraud
cost_true_positive = 10  # Cost of successful fraud intervention

# Use champion model scores (XGBoost in most cases)
champion_scores = xgb_valid_scores  # Update based on actual champion

# Test multiple thresholds
thresholds_to_test = np.arange(0.01, 0.99, 0.01)
costs = []
metrics_by_threshold = []

for threshold in thresholds_to_test:
    preds = (champion_scores >= threshold).astype(int)
    cm = confusion_matrix(y_valid, preds)
    tn, fp, fn, tp = cm.ravel()
    
    # Calculate total cost
    total_cost = (fp * cost_false_positive + 
                  fn * cost_false_negative + 
                  tp * cost_true_positive)
    
    costs.append(total_cost)
    metrics_by_threshold.append({
        'threshold': threshold,
        'total_cost': total_cost,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
        'precision': precision_score(y_valid, preds, zero_division=0),
        'recall': recall_score(y_valid, preds, zero_division=0),
        'f1': f1_score(y_valid, preds, zero_division=0)
    })

# Find optimal threshold (minimum cost)
optimal_idx = np.argmin(costs)
optimal_threshold = thresholds_to_test[optimal_idx]
optimal_cost = costs[optimal_idx]

print(f'=== Cost-Sensitive Threshold Optimization ===')
print(f'Optimal Threshold: {optimal_threshold:.3f}')
print(f'Minimum Expected Cost: ${optimal_cost:,.2f}')

# Metrics at optimal threshold
optimal_metrics = metrics_by_threshold[optimal_idx]
print(f'\\nPerformance at Optimal Threshold:')
print(f'  Precision: {optimal_metrics["precision"]:.3f}')
print(f'  Recall: {optimal_metrics["recall"]:.3f}')
print(f'  F1-Score: {optimal_metrics["f1"]:.3f}')
print(f'  True Positives: {optimal_metrics["tp"]}')
print(f'  False Positives: {optimal_metrics["fp"]}')
print(f'  False Negatives: {optimal_metrics["fn"]}')

# Visualize cost vs threshold
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Cost curve
ax[0].plot(thresholds_to_test, costs, linewidth=2, color='#e74c3c')
ax[0].axvline(optimal_threshold, color='#2ecc71', linestyle='--', linewidth=2, 
              label=f'Optimal: {optimal_threshold:.3f}')
ax[0].set_xlabel('Decision Threshold')
ax[0].set_ylabel('Total Cost ($)')
ax[0].set_title('Total Cost vs Decision Threshold', fontsize=12, fontweight='bold')
ax[0].legend()
ax[0].grid(alpha=0.3)

# Precision-Recall tradeoff
metrics_df = pd.DataFrame(metrics_by_threshold)
ax[1].plot(metrics_df['threshold'], metrics_df['precision'], label='Precision', linewidth=2, color='#3498db')
ax[1].plot(metrics_df['threshold'], metrics_df['recall'], label='Recall', linewidth=2, color='#e74c3c')
ax[1].plot(metrics_df['threshold'], metrics_df['f1'], label='F1-Score', linewidth=2, color='#2ecc71')
ax[1].axvline(optimal_threshold, color='gray', linestyle='--', linewidth=2, alpha=0.5,
              label=f'Optimal: {optimal_threshold:.3f}')
ax[1].set_xlabel('Decision Threshold')
ax[1].set_ylabel('Score')
ax[1].set_title('Precision-Recall Tradeoff', fontsize=12, fontweight='bold')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Apply optimal threshold for evaluation
optimal_valid_preds = (champion_scores >= optimal_threshold).astype(int)
print('\\n=== Classification Report at Optimal Threshold ===')
print(classification_report(y_valid, optimal_valid_preds))

# Update confusion matrix visualization
ConfusionMatrixDisplay.from_predictions(y_valid, optimal_valid_preds, cmap='Blues')
plt.title(f'Confusion Matrix at Optimal Threshold ({optimal_threshold:.3f})', fontweight='bold')
plt.show()

# Final evaluation on test set with optimal threshold
test_scores_champion = xgb_model.predict_proba(X_test)[:, 1]
test_preds_optimal = (test_scores_champion >= optimal_threshold).astype(int)

print('=== Final Test Set Performance ===')
print(f'Threshold Used: {optimal_threshold:.3f}')
print(f'\\nTest ROC-AUC: {roc_auc_score(y_test, test_scores_champion):.4f}')
print(f'Test AUPRC: {average_precision_score(y_test, test_scores_champion):.4f}')
print('\\nClassification Report:')
print(classification_report(y_test, test_preds_optimal))

# Calculate test set costs
cm_test = confusion_matrix(y_test, test_preds_optimal)
tn_test, fp_test, fn_test, tp_test = cm_test.ravel()

test_total_cost = (fp_test * cost_false_positive + 
                   fn_test * cost_false_negative + 
                   tp_test * cost_true_positive)

print(f'\\n=== Business Impact on Test Set ===')
print(f'True Positives (Caught Frauds): {tp_test}')
print(f'False Positives (False Alarms): {fp_test}')
print(f'False Negatives (Missed Frauds): {fn_test}')
print(f'True Negatives (Correct Legit): {tn_test}')
print(f'\\nTotal Operational Cost: ${test_total_cost:,.2f}')
print(f'Fraud Detection Rate: {tp_test/(tp_test+fn_test)*100:.1f}%')
print(f'False Alarm Rate: {fp_test/(fp_test+tn_test)*100:.2f}%')

# Visualize test results
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, test_preds_optimal, ax=ax[0], cmap='Blues')
ax[0].set_title(f'Test Set Confusion Matrix\\n(Threshold={optimal_threshold:.3f})', fontweight='bold')

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, test_scores_champion)
ax[1].plot(fpr, tpr, linewidth=2, color='#3498db', label=f'ROC-AUC={roc_auc_score(y_test, test_scores_champion):.3f}')
ax[1].plot([0, 1], [0, 1], 'k--', label='Random')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC Curve - Test Set', fontweight='bold')
ax[1].legend()
ax[1].grid(alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, test_scores_champion)
ax[2].plot(recall, precision, linewidth=2, color='#e74c3c', label=f'AUPRC={average_precision_score(y_test, test_scores_champion):.3f}')
ax[2].set_xlabel('Recall')
ax[2].set_ylabel('Precision')
ax[2].set_title('Precision-Recall Curve - Test Set', fontweight='bold')
ax[2].legend()
ax[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compile all model scores
models_results = {
    'Logistic Regression': valid_scores,
    'Isolation Forest': valid_anomaly,
    'Random Forest': rf_valid_scores,
    'XGBoost': xgb_valid_scores,
    'Gradient Boosting': gb_valid_scores,
    'Local Outlier Factor': lof_valid_scores
}

# Calculate comprehensive metrics
comparison_data = []
for model_name, scores in models_results.items():
    comparison_data.append({
        'Model': model_name,
        'ROC-AUC': roc_auc_score(y_valid, scores),
        'AUPRC': average_precision_score(y_valid, scores)
    })

comparison_df = pd.DataFrame(comparison_data).sort_values('AUPRC', ascending=False)
print('=== Model Performance Comparison (Validation Set) ===')
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# Metrics bar chart
comparison_df.set_index('Model')[['ROC-AUC', 'AUPRC']].plot(
    kind='barh', ax=ax[0], color=['#3498db', '#e74c3c'], alpha=0.7, edgecolor='black'
)
ax[0].set_xlabel('Score')
ax[0].set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
ax[0].legend(['ROC-AUC', 'AUPRC (Primary Metric)'])
ax[0].grid(axis='x', alpha=0.3)
ax[0].set_xlim(0, 1)

# Precision-Recall curves
for model_name, scores in models_results.items():
    precision, recall, _ = precision_recall_curve(y_valid, scores)
    auprc = average_precision_score(y_valid, scores)
    ax[1].plot(recall, precision, label=f'{model_name} (AUPRC={auprc:.3f})', linewidth=2)

ax[1].set_xlabel('Recall')
ax[1].set_ylabel('Precision')
ax[1].set_title('Precision-Recall Curves Comparison', fontsize=12, fontweight='bold')
ax[1].legend(loc='best', fontsize=9)
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Select champion (highest AUPRC - better for imbalanced data)
champion_name = comparison_df.iloc[0]['Model']
print(f'\n🏆 Champion Model: {champion_name}')
print(f'   AUPRC: {comparison_df.iloc[0]["AUPRC"]:.4f}')
print(f'   ROC-AUC: {comparison_df.iloc[0]["ROC-AUC"]:.4f}')

---

## 🤖 AI LEARNING CHECKPOINT #4: Data Mining

**Pause here!** You've trained and compared multiple models.

### What You Just Learned
✅ Specialized algorithms for rare-event detection  
✅ Handling extreme class imbalance (class_weight, contamination)  
✅ Model comparison using AUPRC (better than ROC-AUC for imbalanced data)  
✅ Cost-sensitive threshold optimization  
✅ 6 different anomaly detection approaches  

### Ask AI to Critique

```
You are a machine learning expert specializing in KDD methodology and anomaly detection.
Review my Data Mining phase for fraud detection.

CONTEXT:
- Class imbalance: 579:1
- Models: Logistic Regression, Isolation Forest, Random Forest, XGBoost, 
  Gradient Boosting, Local Outlier Factor
- Primary metric: AUPRC (Area Under Precision-Recall Curve)

MY RESULTS:
[Paste model comparison table]
[Paste champion model and metrics]

CRITIQUE QUESTIONS:
1. Is my model selection appropriate for this extreme imbalance?
2. How well am I leveraging each algorithm's strengths?
3. Should I try ensemble methods or model stacking?
4. Is AUPRC the right primary metric, or should I use others?
5. How can I improve the cost-sensitive threshold optimization?
6. What about semi-supervised or one-class classification approaches?
7. Should I tune hyperparameters differently for rare events?
8. How do I validate that models generalize to new fraud patterns?

Provide 5-10 actionable improvements.
```

### Document AI Feedback

*[Paste AI insights here]*

### Apply Improvements

**Advanced techniques to try:**
- Ensemble stacking with diverse models
- SMOTE or undersampling strategies
- Hyperparameter tuning with Bayesian optimization
- One-class SVM or Autoencoder approaches
- Calibration techniques
- Feature importance analysis across models

---

**Ready?** Continue to **Interpretation & Evaluation** below.

---

In [ ]:
from joblib import dump
import json
from datetime import datetime

artifacts = Path('../app/artifacts')
artifacts.mkdir(parents=True, exist_ok=True)

# Save champion model and threshold
dump(xgb_model, artifacts / 'credit_fraud_model.joblib')
dump(features, artifacts / 'feature_order.joblib')
dump({'optimal_threshold': optimal_threshold}, artifacts / 'threshold_config.joblib')

print(f'✓ Model saved to {artifacts / "credit_fraud_model.joblib"}')
print(f'✓ Feature order saved')
print(f'✓ Optimal threshold ({optimal_threshold:.3f}) saved')

# Create comprehensive model card
model_card = {
    'model_name': 'Credit Card Fraud Detector',
    'version': '1.0.0',
    'created_date': datetime.now().isoformat(),
    'methodology': 'KDD (Knowledge Discovery in Databases)',
    'algorithm': 'XGBoost Classifier',
    'training_data': {
        'source': 'Kaggle - Credit Card Fraud Detection',
        'n_samples_train': len(X_train),
        'n_samples_valid': len(X_valid),
        'n_samples_test': len(X_test),
        'n_features': len(features),
        'class_distribution': f'{(y_train == 0).sum()} legitimate, {(y_train == 1).sum()} fraud',
        'imbalance_ratio': f'{(y_train == 0).sum() / (y_train == 1).sum():.1f}:1'
    },
    'performance_metrics': {
        'test_roc_auc': float(roc_auc_score(y_test, test_scores_champion)),
        'test_auprc': float(average_precision_score(y_test, test_scores_champion)),
        'test_precision': float(precision_score(y_test, test_preds_optimal)),
        'test_recall': float(recall_score(y_test, test_preds_optimal)),
        'test_f1': float(f1_score(y_test, test_preds_optimal)),
        'optimal_threshold': float(optimal_threshold),
        'fraud_detection_rate': float(tp_test/(tp_test+fn_test))
    },
    'business_requirements': {
        'primary_metric': 'AUPRC (Area Under Precision-Recall Curve)',
        'objective': 'Detect fraudulent transactions with minimal false alarms',
        'cost_parameters': {
            'false_positive_cost': cost_false_positive,
            'false_negative_cost': cost_false_negative,
            'true_positive_cost': cost_true_positive
        }
    },
    'kdd_phases_completed': [
        'Selection: Stratified sampling preserving fraud ratio',
        'Preprocessing: Missing value check, time-ordered splits',
        'Transformation: Feature scaling, temporal features, class balancing',
        'Data Mining: Ensemble models with imbalance handling',
        'Interpretation: Cost-sensitive threshold optimization'
    ],
    'limitations': [
        'Model trained on anonymized PCA features - limited interpretability',
        'Dataset from 2013 - may not reflect current fraud patterns',
        'Extreme class imbalance requires careful threshold tuning',
        'Performance depends on maintaining similar transaction distributions'
    ],
    'ethical_considerations': [
        'Ensure fairness across demographic groups (if identifiable)',
        'Minimize false positives to avoid customer inconvenience',
        'Provide appeals process for blocked transactions',
        'Monitor for adversarial attacks and concept drift'
    ],
    'deployment_notes': {
        'inference_requirements': 'Real-time scoring (<100ms)',
        'monitoring': 'Track daily AUPRC, fraud rate, false positive rate',
        'retraining_frequency': 'Weekly or when drift detected',
        'feature_dependencies': f'{len(features)} PCA-transformed features'
    }
}

with open(artifacts / 'model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2)

print(f'✓ Model card saved to {artifacts / "model_card.json"}')
print(f'\\n📦 All deployment artifacts ready in {artifacts}/')

## Data MiningWe experiment with specialized algorithms suited for rare-event detection.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

features = [col for col in train.columns if col not in {'Class', 'Time', 'Amount'}]
X_train, y_train = train[features], train['Class']
X_valid, y_valid = valid[features], valid['Class']
X_test, y_test = test[features], test['Class']

weighted_logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
weighted_logreg.fit(X_train, y_train)
valid_scores = weighted_logreg.predict_proba(X_valid)[:, 1]
print('LogReg ROC-AUC:', roc_auc_score(y_valid, valid_scores))
print('LogReg AUPRC:', average_precision_score(y_valid, valid_scores))


---

## 🤖 AI LEARNING CHECKPOINT #5: Interpretation & Evaluation

**Final checkpoint!** You've evaluated and deployed your fraud detection model.

### What You Just Learned
✅ Evaluating with business-relevant metrics (fraud detection rate, false alarm rate)  
✅ Cost-benefit analysis with realistic fraud economics  
✅ Threshold optimization for business objectives  
✅ Model card creation with KDD documentation  
✅ Production artifact preparation  

### Ask AI to Critique

```
You are a fraud detection deployment expert with deep KDD methodology knowledge.
Review my Interpretation & Evaluation phase.

CONTEXT:
- Business metrics: Fraud detection rate, false alarm rate, operational cost
- Cost parameters: FP=$5, FN=$200, TP=$10
- Optimal threshold: [paste threshold]

TEST SET RESULTS:
[Paste test set performance metrics]
[Paste business impact analysis]

CRITIQUE QUESTIONS:
1. Is my cost-benefit analysis realistic for fraud detection?
2. How should I interpret the test results for stakeholders?
3. What additional evaluation is needed before production deployment?
4. Are my cost parameters reasonable, or should I refine them?
5. How can I better communicate the precision-recall tradeoff?
6. What monitoring should I implement post-deployment?
7. How do I handle concept drift (evolving fraud patterns)?
8. Are there fairness or bias concerns in fraud detection I should address?
9. What documentation is missing from my model card?

Provide 5-10 actionable improvements for deployment readiness.
```

### Document AI Feedback

*[Paste AI recommendations here]*

### Apply Improvements

**Production checklist:**
- [ ] A/B testing framework
- [ ] Real-time monitoring dashboard (AUPRC, fraud rate)
- [ ] Drift detection alerts
- [ ] Model retraining pipeline
- [ ] Explainability for fraud analysts
- [ ] Appeals process for false positives
- [ ] Compliance documentation
- [ ] Performance benchmarks

---

## 🎉 KDD Process Complete!

You've successfully applied the full KDD methodology to fraud detection!

**What you accomplished:**
✅ Selection: Identified and loaded relevant fraud transaction data  
✅ Preprocessing: Cleaned data and created stratified splits  
✅ Transformation: Engineered features and analyzed temporal patterns  
✅ Data Mining: Trained 6 models specialized for rare-event detection  
✅ Interpretation & Evaluation: Optimized for business value with cost-sensitive thresholds  

**Compare methodologies:**
- Explore [CRISP-DM](../../crisp_dm_telco_churn/notebooks/) for business-driven approach
- Check [SEMMA](../../semma_bank_marketing/notebooks/) for rapid prototyping
- Read [HOW_TO_LEARN_WITH_AI.md](../../HOW_TO_LEARN_WITH_AI.md) for learning strategies

---

In [ ]:
iso_forest = IsolationForest(contamination=0.001, random_state=42)
iso_forest.fit(X_train)
valid_anomaly = -iso_forest.score_samples(X_valid)
print('IsolationForest ROC-AUC:', roc_auc_score(y_valid, valid_anomaly))
print('IsolationForest AUPRC:', average_precision_score(y_valid, valid_anomaly))


## Interpretation & Evaluation**Deliverables**- Confusion matrix at the business-defined threshold.- Cost-sensitive analysis (false positive cost vs fraud catch rate).- SHAP value inspection for Logistic Regression coefficients.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve
import matplotlib.pyplot as plt

threshold = 0.2
valid_preds = (valid_scores >= threshold).astype(int)
ConfusionMatrixDisplay.from_predictions(y_valid, valid_preds)
plt.show()

prec, rec, thresh = precision_recall_curve(y_valid, valid_scores)
plt.plot(rec, prec)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()


In [ ]:
test_scores = weighted_logreg.predict_proba(X_test)[:, 1]
print('Test ROC-AUC:', roc_auc_score(y_test, test_scores))
print('Test AUPRC:', average_precision_score(y_test, test_scores))


### Export ArtifactsSave the best performing model and document assumptions for deployment.

In [ ]:
from joblib import dump
artifacts = Path('../app/artifacts')
artifacts.mkdir(parents=True, exist_ok=True)
dump(weighted_logreg, artifacts / 'credit_fraud_model.joblib')
dump(features, artifacts / 'feature_order.joblib')


### Next Steps- Calibrate threshold with business cost matrix.- Try gradient boosting with undersampling.- Integrate streaming inference plan in FastAPI service.